In [ ]:
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from peft import LoraConfig, PeftModel
from datetime import datetime
from trl import SFTTrainer, SFTConfig
import wandb
import numpy as np

In [ ]:
!pip install -U \
    "transformers>=4.50,<5" \
    "trl>=0.17,<0.23" \
    "peft>=0.14" \
    "bitsandbytes>=0.45" \
    "accelerate>=1.2" \
    datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 32.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0
  Attempting 

In [ ]:
INSTRUCT_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
PROJECT_NAME = 'CustSupportQandA'
RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = PROJECT_NAME+RUN_NAME

DATA_USER = HF_USER = "PranayCh"
DATASET_NAME = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME)["train"]


# First: 80% train, 20% temporary
split = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = split["train"]
temp_dataset = split["test"]

# Split the 20% into validation and test
split2 = temp_dataset.train_test_split(test_size=0.5, seed=42)

validation_dataset = split2["train"]
test_dataset = split2["test"]

README.md: 0.00B [00:00, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset(DATASET_NAME)["train"]

def get_token_length(example):
    text = example["instruction"] + "\n" + example["response"]

    return {
        "token_length": len(tokenizer(text)["input_ids"])
    }


lengths = dataset.map(get_token_length,remove_columns=dataset.column_names)
token_lengths = lengths["token_length"]
print("Number of examples:", len(token_lengths))
print("Minimum:", np.min(token_lengths))
print("Mean:", np.mean(token_lengths))
print("Median:", np.median(token_lengths))
print("90th percentile:", np.percentile(token_lengths, 90))
print("95th percentile:", np.percentile(token_lengths, 95))
print("Maximum:", np.max(token_lengths))


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/26872 [00:00<?, ? examples/s]

Number of examples: 26872
Minimum: 19
Mean: 136.07885531408158
Median: 115.0
90th percentile: 225.0
95th percentile: 267.0
Maximum: 491


In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
instruct_model = AutoModelForCausalLM.from_pretrained(INSTRUCT_MODEL,device_map='auto')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
instruct_model.get_memory_footprint()/10**9

6.174857472

In [ ]:
instruct_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [ ]:
quant_config = BitsAndBytesConfig(load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4")

instruct_model = AutoModelForCausalLM.from_pretrained(
    INSTRUCT_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)


In [ ]:
print("Memory footprint:"+ str(instruct_model.get_memory_footprint() / 1e9:,.2f) +"GB")

Memory footprint: 1.12 GB


In [ ]:
EPOCHS = 2
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
MAX_SEQUENCE_LENGTH = 256

# Hyper-parameters - QLoRA

QUANT_4_BIT = True
LORA_R = 16
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS
LORA_DROPOUT = 0.1

# Hyper-parameters - training

LEARNING_RATE = 0.0001
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# Tracking

#VAL_SIZE = length of validation_dataset
LOG_STEPS = 10
SAVE_STEPS = 250
LOG_TO_WANDB = True


In [ ]:
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()


os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: pranaych (pranaych-not-applicable) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
def format_example(example):
    messages = [
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }

In [ ]:
train_dataset = train_dataset.map(format_example)
validation_dataset = validation_dataset.map(format_example)
test_dataset = test_dataset.map(format_example)

Map:   0%|          | 0/21497 [00:00<?, ? examples/s]

Map:   0%|          | 0/2687 [00:00<?, ? examples/s]

Map:   0%|          | 0/2688 [00:00<?, ? examples/s]

In [ ]:
train = train_dataset
val = validation_dataset
test = test_dataset

if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

In [ ]:
# LoRA Parameters

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

In [ ]:
# Training parameters

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS
)

In [ ]:
fine_tuning = SFTTrainer(
    model=instruct_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters
)

Adding EOS to train dataset:   0%|          | 0/21497 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/21497 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/21497 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/2687 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2687 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2687 [00:00<?, ? examples/s]

In [ ]:
# Fine-tune!
fine_tuning.train()

# Push fine-tuned model to Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
250,0.780600,0.848256,0.815075,650173.000000,0.760984
500,0.684900,0.756807,0.706546,1285696.000000,0.779030
750,0.636600,0.698083,0.657088,1924269.000000,0.791385
1000,0.633200,0.670068,0.656501,2573588.000000,0.797382
1250,0.614400,0.647768,0.626289,3218949.000000,0.802622
1500,0.593000,0.636895,0.603950,3870007.000000,0.804994
1750,0.616200,0.626667,0.612692,4513189.000000,0.807478
2000,0.578800,0.617594,0.611257,5158479.000000,0.809339
2250,0.581400,0.614010,0.609018,5803066.000000,0.810365
2500,0.614400,0.612663,0.610807,6441341.000000,0.810778


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors: 100%|##########| 17.5MB / 17.5MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Saved to the hub: CustSupportQandA2026-09-05_07.38.49


In [ ]:
if LOG_TO_WANDB:
  wandb.finish()

eval/entropy,█▄▃▃▂▁▁▁▁▁
eval/loss,█▅▄▃▂▂▁▁▁▁
eval/mean_token_accuracy,▁▄▅▆▇▇████
eval/num_tokens,▁▂▃▃▄▅▆▆▇█
eval/runtime,██▅▆▄▂▂▁▁▃
eval/samples_per_second,▁▁▄▃▅▆▇██▆
eval/steps_per_second,▁▁▄▃▅▆▇██▆
train/entropy,█▇▅▅▄▂▃▂▂▂▃▂▂▂▃▂▂▁▂▁▂▁▂▁▂▁▂▂▂▂▂▁▁▁▁▁▂▁▁▁
train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇█████
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇████
+5,...


In [ ]:
fine_tuning.model.push_to_hub(
    PROJECT_RUN_NAME,
    private=True,
    commit_message="Training complete"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  91%|#########1| 16.0MB / 17.5MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/PranayCh/CustSupportQandA2026-09-05_07.38.49/commit/d5c316e18b60ff733ffd081829b3a1af8a37b9da', commit_message='Training complete', commit_description='', oid='d5c316e18b60ff733ffd081829b3a1af8a37b9da', pr_url=None, repo_url=RepoUrl('https://huggingface.co/PranayCh/CustSupportQandA2026-09-05_07.38.49', endpoint='https://huggingface.co', repo_type='model', repo_id='PranayCh/CustSupportQandA2026-09-05_07.38.49'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = "PranayCh/CustSupportQandA2026-09-05_07.38.49"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

question = "How can I reset my password?"

output = generator(
    [{"role": "user", "content": question}],
    max_new_tokens=128,
    return_full_text=False
)

print(output[0]["generated_text"])

Device set to use cuda:0


I appreciate your request to reset your password. To regain access to your account, please follow these steps:

1. Start by visiting our platform's "{{Login Page URL}}".
2. Look for the option labeled "{{Forgot Password}}" and click on it.
3. On the subsequent screen, you will be prompted to provide the email address associated with your account.
4. After entering the necessary details, you should receive an email containing instructions on how to reset your password. If you don't see the email in your inbox, please check your spam or junk folders as well.
5. Follow the instructions provided in the email carefully to create a


In [ ]:
test

In [ ]:
results = []
import pandas as pd

for i in range(100):
    question = test[i]["instruction"]
    expected = test[i]["response"]

    output = generator(
        [{"role": "user", "content": question}],
        max_new_tokens=128,
        return_full_text=False
    )

    generated = output[0]["generated_text"]

    results.append({
        "question": question,
        "expected": expected,
        "generated": generated
    })


results_df = pd.DataFrame(results)

results_df.head()


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


need help to take a quick look at bill #00108

EXPECTED:
Positively! I'm here to assist you with taking a quick look at your bill with the number #{{Invoice Number}}. To access your bill, please log in to your account on our website and navigate to the "Billing" section. You should be able to find the bill you're looking for by entering the bill number or selecting the corresponding date range. If you encounter any difficulties or have further questions, feel free to let me know. I'm here to ensure you have a seamless experience!

GENERATED:
I understand your need for assistance in quickly reviewing the details of bill {{Invoice Number}}. Let me guide you through the process:

1. Log in to your account on our website.
2. Navigate to the "Billing" or "Invoices" section.
3. Look for the option to view invoices or search for specific numbers.
4. Enter the invoice number {{Invoice Number}} and click on the corresponding button.

If you encounter any difficulties during this process or have

In [ ]:
results_df.to_csv("testresltsOfFinetuning_data.csv", index=False)